In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.optimize_embedding import *
from scripts.perturbation_distance import *
from scripts.TPS import *
from scripts.evaluation import *

import scvelo as scv
import numpy as np
import umap
from sklearn.manifold import TSNE
import anndata
import pandas as pd

In [ ]:
X_gt = np.array(pd.read_csv("./data/circle/position.csv"))
Y_div_gt = np.array(pd.read_csv("./data/circle/velocity_divergence.csv"))
Y_curl_gt = np.array(pd.read_csv("./data/circle/velocity_curl.csv"))
Y_spiral_gt = np.array(pd.read_csv("./data/circle/velocity_spiral.csv"))

color = np.sum(np.sqrt(X_gt**2),axis=1)

plot_2d(X_gt, color)

In [ ]:
def round_metrics(metrics, digits=3):
    return {k: round(v, digits) for k, v in metrics.items()}


from scripts.optimize_embedding import *

np.random.seed(42)

X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_div_gt + np.random.normal(scale=Y_noise_std, size=Y_div_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

In [ ]:
# Step 1: Prepare Data in an AnnData Object
adata = anndata.AnnData(X=X)
adata.layers["position"] = X
adata.layers["velocity"] = Y
adata.obs["time"] = color
# UMAP embedding
umap_model = umap.UMAP(n_components=2)
adata.obsm["X_umap"] = umap_model.fit_transform(adata.X)

# t-SNE embedding
tsne_model = TSNE(n_components=2)
adata.obsm["X_tsne"] = tsne_model.fit_transform(adata.X)

scv.pp.neighbors(adata)
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")

# UMAP with quiver plot (alternative visualization)
scv.pl.velocity_embedding_grid(adata, basis="umap", color="time", density=0.8, arrow_size=1.5)

plot_2d_quiver(adata.obsm["X_umap"], adata.obsm["velocity_umap"], color, s=30, scale=0.2, alpha=0.6, cmap="viridis")

In [ ]:
print("Raw data")
print(round_metrics(position_embedding_evaluation_metric(X, X_gt)))
print(round_metrics(vector_field_evaluation_metric(Y, Y_div_gt)))

print("ScVelo embedding")
print(round_metrics(position_embedding_evaluation_metric(X_gt, adata.obsm["X_umap"])))
print(round_metrics(vector_field_evaluation_metric(Y_div_gt, adata.obsm["velocity_umap"])))

In [ ]:
np.random.seed(42)

X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_curl_gt + np.random.normal(scale=Y_noise_std, size=Y_curl_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

In [ ]:
# Step 1: Prepare Data in an AnnData Object
adata = anndata.AnnData(X=X)
adata.layers["position"] = X
adata.layers["velocity"] = Y
adata.obs["time"] = color
# UMAP embedding
umap_model = umap.UMAP(n_components=2)
adata.obsm["X_umap"] = umap_model.fit_transform(adata.X)

# t-SNE embedding
tsne_model = TSNE(n_components=2)
adata.obsm["X_tsne"] = tsne_model.fit_transform(adata.X)

scv.pp.neighbors(adata)
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")

# UMAP with quiver plot (alternative visualization)
scv.pl.velocity_embedding_grid(adata, basis="umap", color="time", density=0.8, arrow_size=1.5)

plot_2d_quiver(adata.obsm["X_umap"], adata.obsm["velocity_umap"], color, s=30, scale=0.2, alpha=0.6, cmap="viridis")

In [ ]:
print("Raw data")
print(round_metrics(position_embedding_evaluation_metric(X, X_gt)))
print(round_metrics(vector_field_evaluation_metric(Y, Y_curl_gt)))

print("ScVelo embedding")
print(round_metrics(position_embedding_evaluation_metric(X_gt, adata.obsm["X_umap"])))
print(round_metrics(vector_field_evaluation_metric(Y_curl_gt, adata.obsm["velocity_umap"])))

In [ ]:
np.random.seed(42)

X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_spiral_gt + np.random.normal(scale=Y_noise_std, size=Y_spiral_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

In [ ]:
# Step 1: Prepare Data in an AnnData Object
adata = anndata.AnnData(X=X)
adata.layers["position"] = X
adata.layers["velocity"] = Y
adata.obs["time"] = color
# UMAP embedding
umap_model = umap.UMAP(n_components=2)
adata.obsm["X_umap"] = umap_model.fit_transform(adata.X)

# t-SNE embedding
tsne_model = TSNE(n_components=2)
adata.obsm["X_tsne"] = tsne_model.fit_transform(adata.X)

scv.pp.neighbors(adata)
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")

# UMAP with quiver plot (alternative visualization)
scv.pl.velocity_embedding_grid(adata, basis="umap", color="time", density=0.8, arrow_size=1.5)

plot_2d_quiver(adata.obsm["X_umap"], adata.obsm["velocity_umap"], color, s=30, scale=0.2, alpha=0.6, cmap="viridis")

In [ ]:
print("Raw data")
print(round_metrics(position_embedding_evaluation_metric(X, X_gt)))
print(round_metrics(vector_field_evaluation_metric(Y, Y_spiral_gt)))

print("ScVelo embedding")
print(round_metrics(position_embedding_evaluation_metric(X_gt, adata.obsm["X_umap"])))
print(round_metrics(vector_field_evaluation_metric(Y_spiral_gt, adata.obsm["velocity_umap"])))